<a href="https://colab.research.google.com/github/IrisCheon/NLP-practice/blob/main/Fake_Job_Posting_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fake Job Posting Detection

This project explores the Fake Job Postings dataset using exploratory data analysis and text classification.

The analysis focuses on:
- missing value patterns
- differences between real and fake job postings
- text length characteristics
- metadata features
- Logistic Regression feature importance
- error analysis

The goal is to examine how fake and real job postings differ in both structured fields and textual content, and to identify which patterns are most useful for classification.

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shivamb/real-or-fake-fake-jobposting-prediction")

print("Path to dataset files:", path)

100%|██████████| 16.1M/16.1M [00:00<00:00, 87.8MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/shivamb/real-or-fake-fake-jobposting-prediction/versions/1


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [3]:
os.listdir(path)

['fake_job_postings.csv']

In [7]:
df = pd.read_csv(os.path.join(path, 'fake_job_postings.csv'))
df.head()

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


In [13]:
df.shape

(17880, 18)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17880 entries, 0 to 17879
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   job_id               17880 non-null  int64 
 1   title                17880 non-null  object
 2   location             17534 non-null  object
 3   department           6333 non-null   object
 4   salary_range         2868 non-null   object
 5   company_profile      14572 non-null  object
 6   description          17879 non-null  object
 7   requirements         15184 non-null  object
 8   benefits             10668 non-null  object
 9   telecommuting        17880 non-null  int64 
 10  has_company_logo     17880 non-null  int64 
 11  has_questions        17880 non-null  int64 
 12  employment_type      14409 non-null  object
 13  required_experience  10830 non-null  object
 14  required_education   9775 non-null   object
 15  industry             12977 non-null  object
 16  func

In [10]:
df.columns

Index(['job_id', 'title', 'location', 'department', 'salary_range',
       'company_profile', 'description', 'requirements', 'benefits',
       'telecommuting', 'has_company_logo', 'has_questions', 'employment_type',
       'required_experience', 'required_education', 'industry', 'function',
       'fraudulent'],
      dtype='object')

In [11]:
df["fraudulent"].value_counts()

,count
fraudulent,
0,17014
1,866


# ■ NaN 분석

In [14]:
df.isnull().sum().sort_values(ascending=False)

,0
salary_range,15012
department,11547
required_education,8105
benefits,7212
required_experience,7050
function,6455
industry,4903
employment_type,3471
company_profile,3308
requirements,2696


In [15]:
missing_ratio = ( df.isnull().mean() * 100 )

missing_ratio.sort_values(ascending=False)

,0
salary_range,83.959732
department,64.580537
required_education,45.329978
benefits,40.335570
required_experience,39.429530
function,36.101790
industry,27.421700
employment_type,19.412752
company_profile,18.501119
requirements,15.078300


※ salary_range 와 department는 반 이상의 데이터에서 결측임
- title, job_id, telecommuting, has_questions, has_company_logo, fraudulent는 결측치 없고
- description, location은 거의 대부분의 데이터에 포함

In [22]:
columns = ['location', 'department', 'salary_range',
       'company_profile', 'description', 'requirements', 'benefits', 'employment_type', 'required_experience', 'required_education', 'industry', 'function']

for i in columns:
    print(f"-----{i}-----")
    print(
        pd.crosstab(
            df[i].isnull(),
            df["fraudulent"])
    )

-----location-----
fraudulent      0    1
location              
False       16687  847
True          327   19
-----department-----
fraudulent      0    1
department            
False        5998  335
True        11016  531
-----salary_range-----
fraudulent        0    1
salary_range            
False          2645  223
True          14369  643
-----company_profile-----
fraudulent           0    1
company_profile            
False            14293  279
True              2721  587
-----description-----
fraudulent       0    1
description            
False        17014  865
True             0    1
-----requirements-----
fraudulent        0    1
requirements            
False         14472  712
True           2542  154
-----benefits-----
fraudulent      0    1
benefits              
False       10166  502
True         6848  364
-----employment_type-----
fraudulent           0    1
employment_type            
False            13784  625
True              3230  241
-----required_experience-

    -----salary_range-----
    fraudulent        0    1
    salary_range            
    False          2645  223
    True          14369  643
    -----company_profile-----
    fraudulent           0    1
    company_profile            
    False            14293  279
    True              2721  587
    -----industry-----
    fraudulent      0    1
    industry              
    False       12386  591
    True         4628  275

NaN은 진짜 공고와 가짜 공고에서 얼추 비율이 비슷하지만,
- salary range는 가짜공고의 NaN비율이 확연히 높음(1:3, 진짜는 약 1:7)
- company profile은 진짜 공고에서 NaN이 1/7 수준이었지만, 가짜 공고에서는 NaN값이 우세함(값이 있는 것의 약 2배)
- industry는 진짜 공고에서 NaN 비율은 약 3:1, 가짜공고에서는 약 2:1

# ■ Text Length

In [27]:
df["desc_len"] = df["description"].fillna("").str.len()

In [28]:
df["req_len"] = df["requirements"].fillna("").str.len()
df.sample(5)

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent,desc_len,req_len
16263,16264,Part-time right-hand (start-up NGO),"GR, I, Athens",General Management,NaN,NaN,This is a start-up NGO. So you need to love bo...,Passionate about making the society a better p...,Opportunity to have a true impact to the Greek...,0,1,1,Part-time,Associate,Master's Degree,Nonprofit Organization Management,General Business,0,629,391
902,903,Junior Web / UI Designer,"GR, ,",NaN,NaN,"An independent award winning, ideas-driven, me...",We are seeking canditates who will be exposed ...,"Pursuing a degree in either Graphic Design, De...",NaN,0,1,1,Full-time,Entry level,Some College Coursework Completed,Internet,Advertising,0,950,316
14581,14582,Manufacturing Supervisor,"US, MI, Detroit",NaN,NaN,We Provide Full Time Permanent Positions for m...,(We have more than 1500+ Job openings in our w...,NaN,NaN,0,0,0,Full-time,NaN,NaN,NaN,NaN,0,701,0
15739,15740,Military Veterans Wanted for Recruiting and Sales,"US, CA, Temecula",NaN,NaN,"Empower America is mission driven to recruit, ...",Description:Looking to offer our military vete...,NaN,NaN,0,1,1,Full-time,Not Applicable,NaN,NaN,NaN,0,1528,0
1660,1661,Retail Buyer,"CY, 01, Nicosia",Buying,NaN,NaN,We are seeking for young developable executive...,NaN,NaN,0,0,1,NaN,NaN,NaN,NaN,NaN,0,350,0


In [34]:
df.groupby("fraudulent")[["desc_len", "req_len"]].agg(["mean", "median", "min", "max"])

desc_len                        req_len                  
                   mean  median min    max        mean median min    max
fraudulent                                                              
0           1221.219701  1027.0   6  14907  597.465910  476.5   0  10864
1           1154.834873   844.5   0   8578  446.049654  249.0   0   4077

- desc_len은 글자수 평균은 얼추 비슷하지만
    - median이 진짜 공고가 200자가량 더 높음 > 전반적으로 더 길다
- req_len은 글자수 평균이 가짜 공고에서 446이지만
    - median은 그에 못 미치는 249 > 전반적으로 짧지만, 극단적으로 글자수 편차가 나누어진 것으로 추정
- 결론적으로, 진짜 공고가 description 및 requirements 글자수가 더 길다

#■ fake/real 실제 데이터 비교

In [53]:
real_df = df[df["fraudulent"]==0]
fake_df = df[df["fraudulent"]==1]

In [52]:
real_df[["has_company_logo", "has_questions", "telecommuting"]].mean()

,0
has_company_logo,0.819149
has_questions,0.502057
telecommuting,0.041319


In [50]:
fake_df[["has_company_logo", "has_questions", "telecommuting"]].mean()

,0
has_company_logo,0.326790
has_questions,0.288684
telecommuting,0.073903


- 진짜 공고의 대부분은 company_logo가 있으나 가짜 공고에서는 드뭄(32%)
- questions도 진짜 공고의 절반이 있으나 가짜 공고에서는 드뭄(28%)
- 단, 원격근무는 진짜 공고와 가짜 공고 모두에서 낮은 비율을 차지했으나,
    - 가짜 공고에서의 비율이 진짜 공고에서의 비율의 약 2배

In [55]:
import random as rand
rand.seed(42)
for i in np.random.randint(0, len(fake_df), 5):
    print(fake_df["description"].iloc[i], end="\n\n")

Greetings from VAM SYSTEMS…..VAM SYSTEMS is a Business Consulting, IT Solutions and Services company with operations in UAE, Qatar, Bahrain, USA, Australia, Singapore &amp; India. We offers a comprehensive list of services in the field of IT infrastructure management, Cloud services, IT Consulting Services, Banking, Management Resources, Information Technology Development, Telecom, Aviation, Retail Management &amp; eGovernment offeringsVAM SYSTEMS is currently looking for J2EE Developer for our Bahrain operations .Education Bachelor’s degree in computer science / IT.J2EE Certification.Terms and conditions: Joining time frame:   immediate (maximum 2 weeks)The selected candidates shall join VAM SYSTEMS – Bahrain and shall be deputed to one of the leading Organizations in Bahrain.Should you be interested in this opportunity, please send your latest resume in MS Word format at the earliest at #EMAIL_b08cf5e4101b4b7b7594fe3081f94f7f9a0f2d6192fb5d4a1a72ecf03c816e83#  or call us +91 #PHONE_df

In [58]:
rand.seed(42)
for i in np.random.randint(0, len(real_df), 5):
    print(real_df["description"].iloc[i], end="\n\n")

ACCOUNT MANAGERAs a natural relationship builder you will ensure client satisfaction, retention and identify additional sales opportunities!Our ideal candidate will be responsible for maintaining high levels of client satisfaction, assisting with pre-sales qualification for new opportunities to sell existing clients additional products and services, achieving sales targets for additional products and will be accountable for securing client renewals.Success is driven by exceptional people. We are a growing, dynamic company with a great work environment, opportunities for advancement and leading edge technology. We’re looking for exceptional talent and offer competitive remuneration including benefits and employee recognition. Let’s win together.Your duties would include:Closing business and meeting defined revenue goals within assigned customer portfolio.Targeted prospecting of defined geographical and vertical markets.Fanatical use of #URL_8d92932a488fb7e172d73a0f6813d06d464f1f03705d28

※ 실제 description 텍스트를 봤을 때에는 크게 차이를 잘 모르겠음.

# ■ Logistic Regression

In [84]:
df["requirements"] = df["requirements"].fillna("")
df["company_profile"] = df["company_profile"].fillna("")
df["description"] = df["description"].fillna("")

In [85]:
df["text"] = (
    df["title"] + " " +
    df["company_profile"] + " " +
    df["description"] + " " +
    df["requirements"]
)

In [70]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [93]:
X_train, X_test, y_train, y_test = train_test_split(df["text"], df["fraudulent"], test_size = 0.2, stratify = df["fraudulent"], random_state = 42)

In [78]:
model = Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words = "english",
            max_features = 10000,
            ngram_range= (1,2)
        )),
        ("clf", LogisticRegression(
            max_iter = 1000,
            class_weight = 'balanced'
        ))
])

In [94]:
model.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=10000, ngram_range=(1, 2),
                                 stop_words='english')),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [95]:
y_pred = model.predict(X_test)

In [90]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

In [96]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.98      0.99      3403
           1       0.67      0.89      0.77       173

    accuracy                           0.97      3576
   macro avg       0.83      0.93      0.88      3576
weighted avg       0.98      0.97      0.98      3576



※ 진짜 공고는 잘 탐지하는 것으로 보이지만, 가짜 공고는 잘 탐지하지 못함
- 전체 가짜공고의 89%는 탐지, 단 precision은 67% > FP 과발생
- 모수 크기 자체가 다르다는 문제 때문인 것 같기도 함

#■ coef

In [100]:
feature_names = model["tfidf"].get_feature_names_out()

In [101]:
coefficients = model["clf"].coef_[0]

In [107]:
coef_df = pd.DataFrame({
    "word" : feature_names,
    "coef" : coefficients
}).sort_values("coef", ascending=False)

In [108]:
# fake
coef_df.head(20)

,word,coef
5106,link,4.875900
5700,money,3.911470
2240,data entry,3.890559
638,aptitude staffing,3.703573
9786,work home,3.346074
637,aptitude,3.192155
4140,high school,3.106831
1476,clerk,3.033837
2771,earn,2.848698
715,assistant,2.811945


※ 가짜 공고 상위 20개

    5106	link	4.875900
    5700	money	3.911470
    2240	data entry	3.890559
    638	aptitude staffing	3.703573
    9786	work home	3.346074
    637	aptitude	3.192155
    4140	high school	3.106831
    1476	clerk	3.033837
    2771	earn	2.848698
    715	assistant	2.811945
    2974	engineering	2.738243
    6131	optical	2.721250
    345	administrative	2.702201
    3058	entry	2.636725
    6035	oil gas	2.635455
    4219	hospital	2.614864
    248	accountant	2.594506
    1324	cash	2.554430
    3548	financing	2.500916
    6029	offshore	2.483615

- link가 상위 : "링크를 클릭해라" 는 언급이 가짜뉴스에서 두드러지는 듯
- money, work home, high school, earn 등, 비교적 양적인(객관적인?) 단어들이 상위

In [110]:
# real
coef_df.sort_values("coef", ascending=True).head(20)

,word,coef
8816,team,-3.694053
1665,companies,-3.165310
2990,english,-2.804200
1502,clients,-2.642687
2556,digital,-2.623038
8251,software,-2.558118
7287,recruitment,-2.518884
3960,growing,-2.356793
7849,search,-2.259086
867,based,-2.238441


※ 실제 공고 상위 20개

    8816	team	-3.694053
    1665	companies	-3.165310
    2990	english	-2.804200
    1502	clients	-2.642687
    2556	digital	-2.623038
    8251	software	-2.558118
    7287	recruitment	-2.518884
    3960	growing	-2.356793
    7849	search	-2.259086
    867	based	-2.238441
    5229	love	-2.201283
    9634	web	-2.193564
    5851	new	-2.144708
    7604	right	-2.144694
    3978	growth	-2.126353
    5397	marketing	-2.103426
    3738	fun	-1.984384
    2067	creative	-1.966387
    126	50	-1.949577
    1481	client	-1.928455

- team, companies, clients, growing, love 등
    - 보다 질적인 단어들이 상위인 것이 흥미로움
    - 팀워크, 발전가능성 등등을 강조한다는 느낌



#■ Error Analysis

In [132]:
y_proba = model.predict_proba(X_test)[:,1]

In [135]:
results = pd.DataFrame({
    "text" : X_test,
    "real" : y_test,
    "predict" : y_pred,
    "proba" : y_proba
}).sort_values("proba", ascending=False)

results.head()

,text,real,predict,proba
4832,Senior Engineering Product Manager Aptitude Staffing Solutions has redesigned the recruiting wheel. Our innovative new platform cuts the recruitin...,1,1,0.998936
1821,"Principal/Senior Mechanical Engineer (Package Equipment) Aker Solutions is a global provider of products, systems and services to the oil and gas ...",1,1,0.996164
17672,"Data Entry Clerk As the industry’s largest supply contracting company, Novation serves the purchasing needs of more than 65,000 VHA, UHC and Prov...",1,1,0.993957
2396,"Senior QA Engineer Aptitude Staffing Solutions has redesigned the recruiting wheel. Our innovative new platform cuts the recruiting time in half,...",1,1,0.992697
1813,"Mechanical Assembly & Test Technician Aker Solutions is a global provider of products, systems and services to the oil and gas industry. Our engin...",1,1,0.989924


In [155]:
results["text_length"] = results["text"].str.len()
results.head()

,text,real,predict,proba,text_length
4832,Senior Engineering Product Manager Aptitude Staffing Solutions has redesigned the recruiting wheel. Our innovative new platform cuts the recruitin...,1,1,0.998936,1186
1821,"Principal/Senior Mechanical Engineer (Package Equipment) Aker Solutions is a global provider of products, systems and services to the oil and gas ...",1,1,0.996164,2950
17672,"Data Entry Clerk As the industry’s largest supply contracting company, Novation serves the purchasing needs of more than 65,000 VHA, UHC and Prov...",1,1,0.993957,1384
2396,"Senior QA Engineer Aptitude Staffing Solutions has redesigned the recruiting wheel. Our innovative new platform cuts the recruiting time in half,...",1,1,0.992697,2469
1813,"Mechanical Assembly & Test Technician Aker Solutions is a global provider of products, systems and services to the oil and gas industry. Our engin...",1,1,0.989924,3587


In [156]:
errors = results[(results["real"] != results["predict"])]
errors.head()

,text,real,predict,proba,text_length
16970,Customer Service - Work From Home Do you have 1+ year of customer service/call center experience? Are you looking for flexibility in scheduling a...,0,1,0.957604,1730
6784,Receptionist Quest College is seeking an individual to join the staff as a Receptionist Requirements:Professional demeanorUpbeat and positive at...,0,1,0.856857,537
536,Part-Time Administrative/Data Entry I As Part Time Administrative Assistant/ Data Entry I you will be responsible for:- Reporting directl...,0,1,0.833351,2483
16679,"Urgent Need :Oracle Developer for Bahrain. VAM SYSTEMS is a Business Consulting, IT Solutions and Services company with operations in UAE, Qatar, ...",0,1,0.829282,1429
5428,"Entry Level Legal Jobs in Orange County Great opportunities for entry level legal professionals! Temporary Law Office receptionists, administrati...",0,1,0.828129,988


In [137]:
pd.set_option("display.max_colwidth", 150)

In [157]:
FP_errors = errors[(errors["real"]==0) & (errors["predict"]==1)]
FP_errors.head()

,text,real,predict,proba,text_length
16970,Customer Service - Work From Home Do you have 1+ year of customer service/call center experience? Are you looking for flexibility in scheduling a...,0,1,0.957604,1730
6784,Receptionist Quest College is seeking an individual to join the staff as a Receptionist Requirements:Professional demeanorUpbeat and positive at...,0,1,0.856857,537
536,Part-Time Administrative/Data Entry I As Part Time Administrative Assistant/ Data Entry I you will be responsible for:- Reporting directl...,0,1,0.833351,2483
16679,"Urgent Need :Oracle Developer for Bahrain. VAM SYSTEMS is a Business Consulting, IT Solutions and Services company with operations in UAE, Qatar, ...",0,1,0.829282,1429
5428,"Entry Level Legal Jobs in Orange County Great opportunities for entry level legal professionals! Temporary Law Office receptionists, administrati...",0,1,0.828129,988


In [148]:
for i in FP_errors.head().index:
    print("="*50)
    print(FP_errors.loc[i, "text"])

Customer Service - Work From Home  Do you have 1+ year of customer service/call center experience? Are you looking for flexibility in scheduling and the availability to work from home? We are currently hiring customer service agents with flexible scheduling and you can work from anywhere! This is not a sales position and you do not make outbound calls. Calls you will be taking will be inbound customer service calls!Visit our website at #URL_9b0fb9ec146bbc704d2a4d03360bd38857452f25a28451a579e4a03ec53b08ee# and click on "Get Started" to complete a job application. Looking for a positive change in your career?  Are you ready to start working from home?Job Description:* Assisting customers on inbound calls from your home* Answering questions about service* Walking customers through product/service* Updating customer information Job Requirements/Qualifications:* 1+ year of Customer Service Experience* Quiet home office environment* Computer with Internet Explorer 8 * Headset with microphone

- 첫 번째 텍스트는 다른 FP에 비교하여도 유독 proba가 높음(95.7%)
    - work home, link, high school 등 높은 coef의 단어가 많이 등장
- 다른 텍스트에서는 두드러지는 내용이 없음

In [158]:
FN_errors = errors[(errors["real"]==1) & (errors["predict"]==0)]
FN_errors.tail()

,text,real,predict,proba,text_length
5499,Brand Partner Looking for motivated and hardworking individuals.*Must be at least 18*Learn entrepreneurial and real-life business skills THROUGH ...,1,0,0.143481,490
5672,Software Design Engineer Software Design EngineerQualified candidates are encouraged to apply directly to this job posting. Direct email and pho...,1,0,0.142228,1579
17716,Sales associate Home Security Sales (inside/outside)Data compilation of prospective sales leads. These leads usually come from various sources th...,1,0,0.106102,1247
17644,Fidelity The key to this position is that Steve wants to see how people think. He doesn't want to see extensive models or get a lot of backgroun...,1,0,0.086399,988
3463,Business Development Manager We are currently looking for a client-focused and results-driven Business Development Manager to join our team.Worki...,1,0,0.054449,1064


In [151]:
for i in FN_errors.tail().index:
    print("="*50)
    print(FN_errors.loc[i, "text"])

Brand Partner  Looking for motivated and hardworking individuals.*Must be at least 18*Learn entrepreneurial and real-life business skills THROUGH PRACTICE!Work face to face marketing; become successful while being social!Vemma is a health and nutritional company that is worth over $120 million and is on the rise!If you are looking for a great opportunity to consume very healthy products and get paid to promote them through your friend and family market, contact Zack, space is limited! 
Software Design Engineer  Software Design EngineerQualified candidates are encouraged to apply directly to this job posting.  Direct email and phone calls are not being considered.  Thank you for your cooperation.  Please no recruiters.Job DescriptionTV2 Consulting's is a System Integrator for the worlds #1 TV and IP video platform - Mediaroom. TV2 develops software products that empower Mediaroom operators to deliver the best customer experience.TV2 is looking for a Software Design Engineer to join the 

- FN 아래에서부터(가장 진짜 공고와 유사하다고 판단한 텍스트)
    - 업무에 대한 자세한 설명, 필요한 역량 및 태도 등이 상당히 구체적임.

In [159]:
results["text_length"].describe()

,text_length
count,3576.000000
mean,2500.772651
std,1371.202146
min,82.000000
25%,1554.000000
50%,2344.500000
75%,3206.000000
max,11534.000000


In [160]:
errors["text_length"].describe()

,text_length
count,94.000000
mean,1810.138298
std,1311.863991
min,271.000000
25%,903.500000
50%,1532.000000
75%,2501.000000
max,6482.000000


- 전체 텍스트 대비 error의 텍스트가 전반적으로 더 짧음(평균 700자 정도)
    - 중간값은 900자 가량 차이나며, max는 거의 2배 차이.
- 글자수가 짧은 경우 오탐지 가능성이 높아지는 것으로 보임

In [163]:
FN_errors["text_length"].describe()

,text_length
count,19.000000
mean,1728.105263
std,1549.992362
min,337.000000
25%,781.500000
50%,1247.000000
75%,2094.000000
max,6482.000000


In [162]:
FP_errors["text_length"].describe()

,text_length
count,75.00000
mean,1830.92000
std,1255.51206
min,271.00000
25%,963.00000
50%,1660.00000
75%,2515.50000
max,6477.00000


- FN의 글자수는 전체 테스트데이터와 비교해도 적다
    - 전체: 평균 2500, 중간값 2344 / FN: 평균 1728, 중간값 1247 / FP: 평균 1830, 중간값 1660
    - 데이터 수 자체가 작기 때문에 상대적으로 정확하지 않을 수 있음

# Conclusion

Several metadata fields showed noticeable differences between real and fake job postings. Fake postings were more likely to have missing values in fields such as salary range, company profile, and industry. Real postings were also more likely to include company logos and screening questions.

Text length analysis showed that real job postings generally contained longer descriptions and requirements. Error cases were typically shorter than the overall dataset, suggesting that limited textual information may increase classification difficulty.

Logistic Regression achieved high recall for fake postings but produced a number of false positives. Feature importance analysis suggested that words related to links, earnings, remote work, and educational requirements were more strongly associated with fake postings. In contrast, words related to teams, companies, clients, and growth appeared more frequently among real postings.

Although some patterns were visible in both metadata and model coefficients, manually reading job descriptions showed that the distinction between real and fake postings was not always obvious. Error analysis indicated that shorter texts and postings with limited contextual information were more likely to be misclassified.